In [6]:
from google import genai
from google.genai import types

import os
import time
from pathlib import Path
from dotenv import load_dotenv

# make a ".env" file and fill GEMINI_API_KEY=
# it is gitignored so no worries
load_dotenv() 
client = genai.Client() 

In [3]:
json = {
  "sik_no": int, # the topmost number in the squares
  "voter_count": int, # брой на избирателите - точка 1)
  "additional_voter_count": int, # избиратели под чертата - точка 2)
  "registered_votes": int, # избирателите според положените подписи - точка 3)
  "paper_ballots": {
     "total": int, # тоталния брой хартиени бюлетини в секция - брой на получените бюлетини - точка А
     "unused_ballots": int, # точка 4а
     "registered_vote": int, # намерените бюлетините в кутията с хартии - точка 5)
     "invalid_out_of_the_box": int, # брой на недейстителните хартиени бюлетини за образци - точка 4б
     "invalid_in_the_box": int, # брой на недейстителните хартиени бюлетини в избирателната кутия - точка 6)
     "support_noone": int,
     "votes": [
        {
           "party_number": int, # номера на партията
           "votes": int, # разпределението за хартиените бюлетини общо без преференции - точка 8
           "preferences": [ # да се извлече от - точка 10
             {
                "candidate_number": int, # например 101
                "count": int # колко са гласували с тази преференция
             }
           ],
           "no_preferences": int # без преференция - точка 10
        },
        {
           "...": "..."
        }
     ],
     "total_valid_votes": int # общ брой на действителните гласове - точка 9
  },
  "machine_ballots": {
     "total_votes": int, # брой избиратели от машинно гласуване - точка 11 
     "support_noone": int,  # не подкрепям никого точка 12
     "total_valid_votes": int, # точка 14
     "votes": [
        {
           "party_number": int, # номер на партията - точка 13
           "votes": int, # действителни гласове - точка 13
           "preferences": [  # да се извлече от - точка 15
             {
               "candidate_number": int, # например 103
               "count": int # колко са гласували с тази преференция
             }
           ],
           "no_preferences": int # без преференция - точка 10
        },
        {
           "...": "..."
        }
     ]
  }
}

In [4]:
prompt = f"""
Ти си експерт по изборни одити. Разгледай този сканиран протокол, написан на ръка.
Намери и извлечи числата за следните конкретни полета от този JSON:
{json}
Ако дадено число е било задраскано, зачеркнато с черта или е напълно неразбираемо, върни -1. Върни въпросния JSON с пълните полета.
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash", 
    contents=[uploaded_file, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json", 
        temperature=0.0
    )
)

print("\n--- Extracted Protocol Data ---")
print(response.text)

In [7]:
INPUT_FOLDER = Path("/home/exvick/Desktop/Hackathon/data/input/source_templates")
OUTPUT_FOLDER = Path("/home/exvick/Desktop/Hackathon/data/output/extracted_jsons")

# Create the output folder if it doesn't exist yet
OUTPUT_FOLDER.mkdir(exist_ok=True)

In [ ]:
pdf_files = list(INPUT_FOLDER.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs to process.\n")

Found 9 PDFs to process.



In [9]:
for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}...")
    uploaded_file = None
    
    try:
        # 1. Upload the file
        uploaded_file = client.files.upload(file=str(pdf_path))
        
        # 2. Run Inference
        response = client.models.generate_content(
            model="gemini-2.5-flash", 
            contents=[uploaded_file, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json", 
                temperature=0.0
            )
        )
        
        # 3. Save the JSON output
        # .stem gets the filename without the .pdf extension (e.g., "scan_1")
        output_file_path = OUTPUT_FOLDER / f"{pdf_path.stem}.json"
        
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(response.text)
            
        print(f"Saved to {output_file_path.name}")

    except Exception as e:
        # If anything goes wrong, print the error but KEEP GOING to the next file
        print(f"Failed to process {pdf_path.name}. Error: {e}")
        
    finally:
        # 4. Clean up the file from Google's servers to save space
        if uploaded_file:
            try:
                client.files.delete(name=uploaded_file.name)
            except Exception as cleanup_error:
                print(f"Warning: Could not delete {uploaded_file.name}: {cleanup_error}")
    
    # 5. Rate Limiting Pause
    # Pause for 3 seconds between files to avoid hitting the free API limits
    time.sleep(3)

Processing: 321180621.0.pdf...
Saved to 321180621.0.json
Processing: 321180673.0.pdf...
Saved to 321180673.0.json
Processing: 234610112.0.pdf...
Saved to 234610112.0.json
Processing: 320490218.0.pdf...
Saved to 320490218.0.json
Processing: 122400005.0.pdf...
Saved to 122400005.0.json
Processing: 122900056.0.pdf...
Saved to 122900056.0.json
Processing: 171300055.0.pdf...
Saved to 171300055.0.json
Processing: 273100100.0.pdf...
Saved to 273100100.0.json
Processing: 181700015.0.pdf...
Saved to 181700015.0.json
